In [0]:
%pip install psycopg2-binary

#Ingested operational database sources via JDBC/Python connector into Bronze Delta Tables - PostgreSQL (Render)

In [0]:
import psycopg2
import pandas as pd
from pyspark.sql.functions import current_timestamp, lit 

#Connection config using secrets 

In [0]:
host     = dbutils.secrets.get("jdbc", "pg-host")
user     = dbutils.secrets.get("jdbc", "pg-user")
password = dbutils.secrets.get("jdbc", "pg-password")
database = dbutils.secrets.get("jdbc", "pg-database")

conn = psycopg2.connect(
    host=host, port=5432, database=database,
    user=user, password=password
)

#Tables to ingest

In [0]:
TABLES = [
    "crm_cust_info",
    "crm_prd_info", 
    "crm_sales_details",
    "erp_cust_az12",
    "erp_loc_a101",
    "erp_px_cat_g1v2"
]

#Ingest each table into Bronze

In [0]:
for table in TABLES:
    print(f"Ingesting {table}...")
    pdf = pd.read_sql(f"SELECT * FROM {table}", conn)
    df = (spark.createDataFrame(pdf)
              .withColumn("ingested_at", current_timestamp())
              .withColumn("source", lit("postgresql")))
    df.write.mode("overwrite").format("delta").saveAsTable(f"workspace.bronze.jdbc_{table}")
    print(f"Done — {df.count()} rows loaded into workspace.bronze.jdbc_{table}")

conn.close()
print("All tables ingested successfully!")